Industry-Style Agentic RAG with LangGraph

Why this is industry-style

A simple RAG system searches only the private knowledge base.

An industry-style Agentic RAG system should do this:

Question

↓

Search private knowledge base first

  ↓

Grade the private evidence

  ↓

If private evidence is good → answer from KB

  ↓

If private evidence is weak → search the web

  ↓

Grade web evidence

  ↓

Generate a grounded answer with source type

This design is useful because private documents may be incomplete or outdated.

1.Install Dependencies

2.Configure API Keys

In [5]:
import os
from getpass import getpass
try:
   from dotenv import load_dotenv
   load_dotenv()
except Exception:
   pass
if not os.getenv("CLAUDE_API_KEY"):
   os.environ ["CLAUDE_API_KEY"] = getpass("Enter CLAUDE_API_KEY: ")
if not os.getenv("TAVILY_API_KEY"):
   os.environ["TAVILY_API_KEY"] = getpass("Enter TAVILY_API_KEY: ")
if not os.getenv("PINECONE_API_KEY"):
    os.environ ["PINECONE_API_KEY"] = getpass("Enter PINECONE_API_KEY: ")
print("CLAUDE_API_KEY configured:", bool(os.getenv("CLAUDE_API_KEY")))
print("TAVILY_API_KEY configured:", bool(os.getenv("TAVILY_API_KEY")))
print("PINECONE_API_KEY configured:", bool(os.getenv("PINECONE_API_KEY")))

CLAUDE_API_KEY configured: True
TAVILY_API_KEY configured: True
PINECONE_API_KEY configured: True


3.Load Reaal Documents

We will create our private knowledge base from a real web document:

https://docs.langchain.com/oss/python/langgraph/agentic-rag

This becomes our internal/private KB for the Agentic-RAG.

The web fallback is separate. That means:

   .First source: indexed LangGraph documentation page
   
   .Fallback source: live web search through Tavily


In [6]:
from langchain_community.document_loaders import WebBaseLoader

SOURCE_URL = "https://docs.langchain.com/oss/python/langgraph/agentic-rag"

loader = WebBaseLoader(
    web_paths=(SOURCE_URL,),
    requests_kwargs={
        "headers": {
            "User-Agent": "Mozilla/5.0 Agentic-RAG-Industry-Demo"
        }
    },
)

raw_docs = loader.load()

print("Loaded documents:", len(raw_docs))
print("Source:", raw_docs[0].metadata.get("source"))
print("\nPreview:\n")
print(raw_docs[0].page_content[:1500])

C:\Users\sivab\AppData\Local\Temp\ipykernel_17192\625167974.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded documents: 1
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag

Preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL assistantLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextHuman-in-the-loopGraph APIFunctional AP

4.Split the Docuents into chunks

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True,
)

chunks = splitter.split_documents(raw_docs)

print("Total chunks:", len(chunks))
print("\nFirst chunk preview:\n")
print(chunks[0].page_content[:900])

Total chunks: 28

First chunk preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL assistantLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextHuman-in-the-loopGraph APIFunctional A


5. Create Free Local Embeddings

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

sample_vector = embeddings.embed_query("What is Agentic RAG?")
print("Embedding dimensions:", len(sample_vector))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6826.90it/s]


Embedding dimensions: 384


6. Create the Pinecone Vector Database

The embedding model used in this notebook produces 384-dimensional vectors, so the Pinecone index must use dimension 384.

In [9]:
import os
import time
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# Use a new index name to avoid the existing dimension mismatch
INDEX_NAME = "industry-agentic-rag-384"
NAMESPACE = "langgraph-agentic-rag"

# Connect to Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Check existing indexes
existing_indexes = [
    index_info["name"] for index_info in pc.list_indexes()
]

# Create the index if it does not exist
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

    # Wait until the index is ready
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

else:
    # Verify the existing index dimension
    index_info = pc.describe_index(INDEX_NAME)
    if index_info.dimension != 384:
        raise ValueError(
            f"Index dimension is {index_info.dimension}, "
            "but embeddings require 384. Choose another index name."
        )

print("Pinecone index ready:", INDEX_NAME)

# Upload chunks and create the vector store
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
)

# Create the retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

print("Pinecone vector database and retriever are ready.")

Pinecone index ready: industry-agentic-rag-384
Pinecone vector database and retriever are ready.


In [21]:
# Load Existing Index

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "industry-agentic-rag-kb"
NAMESPACE = "langgraph-agentic-rag"

# Connect to Pinecone
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Load existing Pinecone index
index = pc.Index(INDEX_NAME)

# Connect existing index with LangChain
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE,
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 4,
        "namespace": NAMESPACE,
    }
)

print("Existing Pinecone index loaded:", INDEX_NAME)

Existing Pinecone index loaded: industry-agentic-rag-kb



7. Test Private KB Retrieval

In [24]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(
    index_name="industry-agentic-rag-384",
    embedding=embeddings,
    namespace="langgraph-agentic-rag"
)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created successfully!")

Retriever created successfully!


In [25]:
# 7. Test Private KB Retrieval

test_question = "What happens if retrieved documents are not relevant in Agentic RAG?"

kb_docs = retriever.invoke(test_question)

for i, doc in enumerate(kb_docs, 1):
    print(f"\n--- KB RESULT {i} ---")

    print("Source:", doc.metadata.get("source"))

    print(doc.page_content[:700])


--- KB RESULT 1 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsData AnalysisDeep ResearchContent BuilderRAGLangChainDeep AgentSemantic searchSQL agentVoice agentMulti-agentSubagents: Personal assistantHandoffs: Customer supportRouter: Knowledge baseSkills: SQL as

--- KB RESULT 2 ---
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: 

8. Initialize Groq LLM

In [5]:
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic

load_dotenv(override=True)

api_key = os.getenv("CLAUDE_API_KEY")

if not api_key:
    raise ValueError("CLAUDE_API_KEY not found in .env file")

llm = ChatAnthropic(
    model="claude-sonnet-4-5",
    temperature=0,
    max_tokens=1024,
    api_key=api_key
)

try:
    response = llm.invoke(
        "Explain Agentic RAG in one sentence."
    )
    print(response.content)

except Exception as e:
    print("Error:", repr(e))
    if hasattr(e, "body"):
        print("Error details:", e.body)

Error: AnthropicInvalidRequestError("Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CgbcBH3UKxioSU9k85zHQ'}")
Error details: {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CgbcBH3UKxioSU9k85zHQ'}


In [1]:
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic

load_dotenv()

llm = ChatAnthropic(
    model="claude-sonnet-4-5",
    temperature=0,
    api_key=os.getenv("CLAUDE_API_KEY")
)

response = llm.invoke("Explain Agentic RAG in one sentence.")

print(response.content)

AnthropicInvalidRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CgbbfiHSdqYcVfPiBfuo8'}